In [ ]:
%%capture
!pip install --proxy=192.168.2.10:8080 --upgrade snowflake-connector-python[pandas]
!pip install --proxy=192.168.2.10:8080 --upgrade keyring seaborn statsmodels

In [ ]:
import os
import json
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import snowflake.connector

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)

import warnings
warnings.filterwarnings('ignore')

# Setting Up the Connector

In [ ]:
with open('./credentials.json', 'r') as config_file:
    config = json.load(config_file)

snowflake_creds = config['snowflake_credentials']

ctx = snowflake.connector.connect(**snowflake_creds)
cur = ctx.cursor()

# Load Datasets

In [ ]:
cur.execute("""
SELECT * FROM IRONONETECH_DB.PUBLIC.V_FORECAST_CURVE_CONSOLIDATED;
""")
df = cur.fetch_pandas_all()
df.head()

# Data Preparation

In [ ]:
portfolio       = 'Buckeye'
sub_portfolio   = 'PP'
last_input_date = '2024-12-31'
TARGET_METRIC   = 'Ending Outstanding Loans'

In [ ]:
portfolio_df = df[df['PORTFOLIO'] == portfolio]
portfolio_df = portfolio_df[portfolio_df['SUB_PORTFOLIO'] == sub_portfolio]
portfolio_df = portfolio_df[portfolio_df['FORECAST_TYPE'] == 'Actual']

dff = portfolio_df.groupby(['DATE', 'METRIC']).sum().reset_index()

data_df = dff.pivot(
    index='DATE',
    columns='METRIC',
    values='METRIC_VALUE',
).sort_index()

train_df = data_df[data_df.index <= datetime.date.fromisoformat(last_input_date)]
test_df  = data_df[data_df.index >  datetime.date.fromisoformat(last_input_date)]

print(f"Train: {len(train_df)} rows  ({train_df.index[0]} → {train_df.index[-1]})")
print(f"Test : {len(test_df)} rows  ({test_df.index[0]} → {test_df.index[-1]})")
train_df.head()

# Helpers

In [ ]:
def smape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(
        np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred))) * 100
    )


def print_metrics(label: str, y_true: np.ndarray, y_pred: np.ndarray) -> None:
    print(f"  {label}")
    print(f"    MAPE  : {mean_absolute_percentage_error(y_true, y_pred) * 100:.2f}%")
    print(f"    SMAPE : {smape(y_true, y_pred):.2f}%")
    print(f"    MAE   : {mean_absolute_error(y_true, y_pred):>14,.0f}")
    print(f"    RMSE  : {np.sqrt(mean_squared_error(y_true, y_pred)):>14,.0f}")


def plot_results(pred_df: pd.DataFrame, metric: str) -> None:
    palette = {'Actual': '#2196F3', 'Prediction': '#FF5722', '2025 0+12': '#4CAF50'}
    order   = ['Actual', 'Prediction', '2025 0+12']
    existing = [k for k in order if k in pred_df['FORECAST_TYPE'].unique()]

    fig, ax = plt.subplots(figsize=(20, 6))
    ax.set_title(metric, fontsize=14, fontweight='bold')

    for label in existing:
        subset = pred_df[pred_df['FORECAST_TYPE'] == label].sort_values('DATE')
        ax.plot(
            range(len(subset)),
            subset['METRIC_VALUE'].values,
            marker='o', label=label,
            color=palette.get(label, None), linewidth=2,
        )
        for i, (_, row) in enumerate(subset.iterrows()):
            ax.annotate(
                f"{row['METRIC_VALUE']:,.0f}",
                (i, row['METRIC_VALUE']),
                textcoords='offset points', xytext=(0, 8),
                ha='center', fontsize=7,
            )

    x_labels = (
        pred_df[pred_df['FORECAST_TYPE'] == existing[0]]
        .sort_values('DATE')['DATE'].astype(str).tolist()
    )
    ax.set_xticks(range(len(x_labels)))
    ax.set_xticklabels(x_labels, rotation=90)
    ax.set_xlabel('DATE')
    ax.set_ylabel('METRIC_VALUE')
    ax.legend()
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:,.0f}'))
    plt.tight_layout()
    plt.show()


def get_benchmark(test_dates):
    """Pull 2025 0+12 values for the test period."""
    mask = (
        (df['PORTFOLIO']     == portfolio) &
        (df['SUB_PORTFOLIO'] == sub_portfolio) &
        (df['METRIC']        == TARGET_METRIC) &
        (df['FORECAST_TYPE'] == '2025 0+12')
    )
    bdf = (
        df[mask][['DATE', 'METRIC_VALUE']]
        .copy()
        .assign(
            DATE=lambda x: pd.to_datetime(x['DATE']).dt.date,
            FORECAST_TYPE='2025 0+12',
        )
    )
    return bdf[bdf['DATE'].isin(test_dates)]


def build_pred_df(forecast_values, test_series):
    """Combine Actual, Prediction and 2025 0+12 into a long-format DataFrame."""
    actual_rows = pd.DataFrame({
        'DATE'         : test_series.index,
        'FORECAST_TYPE': 'Actual',
        'METRIC_VALUE' : test_series.values,
    })
    pred_rows = pd.DataFrame({
        'DATE'         : test_series.index,
        'FORECAST_TYPE': 'Prediction',
        'METRIC_VALUE' : forecast_values,
    })
    bench_rows = get_benchmark(test_series.index)
    pred_df = pd.concat([actual_rows, pred_rows, bench_rows[['DATE','FORECAST_TYPE','METRIC_VALUE']]], ignore_index=True)
    pred_df['DATE'] = pred_df['DATE'].apply(lambda x: x.date() if isinstance(x, pd.Timestamp) else x)
    return pred_df


def evaluate_pred_df(pred_df: pd.DataFrame, model_label: str) -> None:
    y_actual = (
        pred_df[pred_df['FORECAST_TYPE'] == 'Actual']
        .sort_values('DATE').set_index('DATE')['METRIC_VALUE']
    )
    print(f"\n{'='*55}")
    print(f"  Error Metrics — {TARGET_METRIC}")
    print(f"{'='*55}")
    for col in [model_label, '2025 0+12']:
        subset = pred_df[pred_df['FORECAST_TYPE'] == col]
        if subset.empty:
            print(f"\n  {col}  — no data")
            continue
        y_pred = subset.sort_values('DATE').set_index('DATE')['METRIC_VALUE']
        common = y_actual.index.intersection(y_pred.index)
        if not len(common):
            print(f"\n  {col}  — no overlapping dates")
            continue
        yt = y_actual[common].values.astype(float)
        yp = y_pred[common].values.astype(float)
        print_metrics(col, yt, yp)
    print(f"{'='*55}")

# Model 1 — SARIMA(1,1,0)(0,1,1,12)

**Non-seasonal** `(1,1,0)`: PACF cuts off after lag 1 (p=1); first-difference removes downward trend (d=1); ACF decays slowly so no MA term (q=0).  
**Seasonal** `(0,1,1,12)`: m=12 annual cycle; D=1 strips the repeating December/June shock; Q=1 smooths leftover seasonal noise after differencing.

In [ ]:
train_y = train_df[TARGET_METRIC].astype(float)
test_y  = test_df[TARGET_METRIC].astype(float)

sarima_model = SARIMAX(
    endog  = train_y,
    order  = (1, 1, 0),
    seasonal_order = (0, 1, 1, 12),
    trend  = 'n',
    enforce_stationarity  = False,
    enforce_invertibility = False,
)
sarima_result = sarima_model.fit(disp=False)
print(sarima_result.summary())

sarima_fc   = sarima_result.get_forecast(steps=len(test_y)).predicted_mean
sarima_fc.index = test_y.index

pred_df_sarima = build_pred_df(sarima_fc.values, test_y)
pred_df_sarima.loc[pred_df_sarima['FORECAST_TYPE'] == 'Prediction', 'FORECAST_TYPE'] = 'SARIMA'
pred_df_sarima  # rename back for display


In [ ]:
plot_results(pred_df_sarima, f'SARIMA(1,1,0)(0,1,1,12) — {TARGET_METRIC}')
evaluate_pred_df(pred_df_sarima, 'SARIMA')

# Model 2 — Holt-Winters (Triple Exponential Smoothing)

**Trend = `'add'`** — the portfolio balance decays by a roughly fixed dollar amount each month (straight linear slope), so additive trend is the right fit.  
**Seasonality = `'add'`** — the December spike and June dip are a stable fixed-dollar swing, not a multiplying percentage, so additive seasonality applies.  
**Seasonal periods = `12`** — locks onto the 12-month annual rhythm.

In [ ]:
hw_model = ExponentialSmoothing(
    train_y,
    trend            = 'add',
    seasonal         = 'add',
    seasonal_periods = 12,
)
hw_result = hw_model.fit(optimized=True)
print(hw_result.summary())

hw_fc = hw_result.forecast(steps=len(test_y))
hw_fc.index = test_y.index

pred_df_hw = build_pred_df(hw_fc.values, test_y)
pred_df_hw.loc[pred_df_hw['FORECAST_TYPE'] == 'Prediction', 'FORECAST_TYPE'] = 'Holt-Winters'
pred_df_hw

In [ ]:
plot_results(pred_df_hw, f'Holt-Winters (A,A,12) — {TARGET_METRIC}')
evaluate_pred_df(pred_df_hw, 'Holt-Winters')

# Model 3 — ETS(A, A, A)

State-space formulation of exponential smoothing.  
**Error = A** — residuals stay roughly constant (no expanding variance).  
**Trend = A** — linear downward slope.  
**Seasonality = A** — fixed-dollar seasonal swings.  

In `statsmodels`, implemented as `ExponentialSmoothing(trend='add', seasonal='add', seasonal_periods=12)` with `initialization_method='estimated'`.

In [ ]:
ets_model = ExponentialSmoothing(
    train_y,
    trend                = 'add',
    seasonal             = 'add',
    seasonal_periods     = 12,
    initialization_method= 'estimated',
)
ets_result = ets_model.fit(optimized=True)
print(ets_result.summary())

ets_fc = ets_result.forecast(steps=len(test_y))
ets_fc.index = test_y.index

pred_df_ets = build_pred_df(ets_fc.values, test_y)
pred_df_ets.loc[pred_df_ets['FORECAST_TYPE'] == 'Prediction', 'FORECAST_TYPE'] = 'ETS'
pred_df_ets

In [ ]:
plot_results(pred_df_ets, f'ETS(A,A,A) — {TARGET_METRIC}')
evaluate_pred_df(pred_df_ets, 'ETS')